# GLI3

Cohort: `GLI3`, Phenopacket Store v0.1.27 (n = 81).

- Greig cephalopolysyndactyly syndrome (GCPS), [OMIM:175700](https://omim.org/entry/175700), n = 50
- Pallister-Hall syndrome (PHS), [OMIM:146510](https://omim.org/entry/146510), n = 21
- Postaxial polydactyly, types A1 and B, [OMIM:174200](https://omim.org/entry/174200), n = 10

GCPS and PHS are distinct entities caused by variants in the same gene, and the split is
predictable from variant type and position: haploinsufficiency causes GCPS, whereas
truncating variants in the middle third of the gene, 3' of the zinc-finger domain, cause
PHS (Johnston et al. 2005, PMID 15739154; Demurger et al. 2015, Eur J Hum Genet).

In [ ]:
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

OUTDIR = Path("../case_studies/GLI3")
OUTDIR.mkdir(parents=True, exist_ok=True)

COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

In [ ]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GLI3"))

print(f"Loaded {len(phenopackets)} phenopackets")

In [ ]:
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [ ]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

## Pairwise correlation

In [ ]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.results_table
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

In [ ]:
results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_GLI3.csv")
)

In [ ]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

In [ ]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GLI3",
)

## Synergy: Pallister-Hall vs other GLI3

In [ ]:
condition = dataset.get_condition(
    has_disease("OMIM:146510"),
    name="Pallister-Hall vs other GLI3",
)

condition.value_counts(dropna=False)

In [ ]:
# lowered from the default of 30, the cohort splits 21/60
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=20,
)
synergy_phs = synergy_analyzer.compute_synergy_matrix(condition=condition)

st = synergy_phs.results_table
print(
    f"rows={len(st)} | synergy {st['synergy'].min():.4f} to {st['synergy'].max():.4f}"
    f" | positive={(st['synergy'] > 0).sum()} negative={(st['synergy'] < 0).sum()}"
)
st.head()

In [ ]:
# save_synergy_results() drops everything below synergy >= 0.01,
# so keep the unfiltered table for the dissynergy side
synergy_phs.results_table.to_csv(
    OUTDIR / "synergy_full_GLI3_OMIM_146510.csv", index=False
)
synergy_phs.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GLI3_OMIM_146510.csv")
)

In [ ]:
synergy_phs.plot_synergy_heatmap()

In [ ]:
print("--- top synergy ---")
display(st.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(st.nsmallest(10, "synergy")[COLS])

## Reverse contrast: GCPS vs other GLI3

Better balanced split (50/31).

In [ ]:
condition_gcps = dataset.get_condition(
    has_disease("OMIM:175700"),
    name="GCPS vs other GLI3",
)
synergy_gcps = synergy_analyzer.compute_synergy_matrix(condition=condition_gcps)

sg = synergy_gcps.results_table
sg.to_csv(OUTDIR / "synergy_full_GLI3_OMIM_175700.csv", index=False)
synergy_gcps.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GLI3_OMIM_175700.csv")
)

print(
    f"rows={len(sg)} | synergy {sg['synergy'].min():.4f} to {sg['synergy'].max():.4f}"
)

In [ ]:
synergy_gcps.plot_synergy_heatmap()

In [ ]:
print("--- top synergy ---")
display(sg.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(sg.nsmallest(10, "synergy")[COLS])